### Conexión Base de Datos

In [3]:
import pyodbc
import pandas as pd
server = 'localhost\\SQLEXPRESS'
database = 'AccidentesDB'
use_windows_auth = True

In [4]:
try:
    conn = pyodbc.connect(
        f'DRIVER={{ODBC Driver 17 for SQL Server}};'
        f'SERVER={server};'
        f'DATABASE={database};'
        f'Trusted_Connection=yes;'
    )
    print("Conexión exitosa")
    conn.close()
except Exception as e:
    print("Error:", e)

Conexión exitosa


In [7]:
from src.datos.GestorDatos import GestorDatos

# Ruta de tu archivo CSV original
ruta_csv = r"C:\prediccion_accidentes\data\raw\BD.csv"

# Crear instancia y limpiar datos
gestor = GestorDatos(ruta_csv)
df = gestor.pipeline()   # Esto carga, limpia y devuelve el DataFrame

print("DataFrame listo. Shape:", df.shape)

Archivo cargado: 104898 filas, 21 columnas
Columnas y texto limpiados.
Nulos manejados.
Día/Mes estandarizados.
DataFrame listo. Shape: (104898, 21)


In [8]:
from sqlalchemy import create_engine
import urllib

# Datos de conexión
server = 'localhost\\SQLEXPRESS'
database = 'AccidentesDB'
driver = '{ODBC Driver 17 for SQL Server}'

params = urllib.parse.quote_plus(
    f"DRIVER={driver};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
df.to_sql('accidentes', engine, if_exists='replace', index=False)
print(" Datos guardados correctamente en SQL Server")

 Datos guardados correctamente en SQL Server


### Guardarlo limpio

In [9]:
# Crear instancia con la ruta de tu CSV original
gestor = GestorDatos(r"C:\prediccion_accidentes\data\raw\BD.csv", sep=';', encoding='utf-8')

# Ejecutar limpieza y obtener DataFrame limpio
df_limpio = gestor.pipeline()

# Exportar a CSV limpio
gestor.exportar_csv(r"C:\prediccion_accidentes\data\processed\accidentes_limpio.csv")

Archivo cargado: 104898 filas, 21 columnas
Columnas y texto limpiados.
Nulos manejados.
Día/Mes estandarizados.
CSV exportado a: C:\prediccion_accidentes\data\processed\accidentes_limpio.csv


### Pruebas consultas


In [10]:
df_check = pd.read_sql("SELECT TOP 5 * FROM accidentes", engine)
df_check.head()

,Clase de accidente,Tipo de accidente,Año,Hora,Hora recodificada,Provincia,Cantón,Distrito,Ruta,Kilómetro,...,Calzada vertical,Calzada horizontal,Tipo de calzada,Tipo de circulación,Estado del tiempo,Estado de la calzada,Región Mideplan,Tipo ruta,Día,Mes
0,Solo heridos leves,Salió de la vía,2024,20:00-20:59,18:00-23:59,Limón,Pococí,Guápiles,32,43,...,Pendiente,Curva,Asfalto,Lateral igual sentido,Lluvia,Resbaladiza,Huetar caribe,Nacional,Sábado,Abril
1,Solo heridos leves,Colisión con motocicleta,2024,06:00-06:59,06:00-11:59,Heredia,San Isidro,San José,32,12,...,Plano,Recta,Asfalto,Angulo recto,Buen tiempo,Buena,Central,Nacional,Martes,Mayo
2,Con muertos o graves,Vuelco,2024,15:00-15:59,12:00-17:59,San José,Vázquez de Coronado,Dulce Nombre de Jesús,32,21,...,Pendiente,Curva,Asfalto,Lateral igual sentido,Buen tiempo,Buena,Central,Nacional,Viernes,Mayo
3,Solo heridos leves,Colisión entre vehículos,2024,08:00-08:59,06:00-11:59,Heredia,Santo Domingo,San Miguel,32,7,...,Plano,Recta,Asfalto,Por detrás,Buen tiempo,Buena,Central,Nacional,Martes,Julio
4,Solo heridos leves,Colisión con motocicleta,2024,07:00-07:59,06:00-11:59,Heredia,Santo Domingo,San Miguel,32,5,...,Plano,Recta,Asfalto,Lateral igual sentido,Buen tiempo,Buena,Central,Nacional,Martes,Agosto


In [17]:
query2 = "SELECT Año, COUNT(*) AS cantidad FROM accidentes GROUP BY Año ORDER BY Año"
df2 = pd.read_sql(query2, engine)
print("\n2. Accidentes por año:")
print(df2)


2. Accidentes por año:
    Año  cantidad
0  2018     14742
1  2019     14861
2  2020     11791
3  2021     14150
4  2022     15629
5  2023     16675
6  2024     17050
